# 07.05 · Functional API

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 07.02–07.04. |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Write** a workflow with `@entrypoint` and `@task`: plain Python control flow, futures, parallel tasks.
2. **Use** a checkpointer with the Functional API: resume after an error **without** redoing finished tasks; `previous` and `entrypoint.final` for short-term memory.
3. **Follow** the rules that make this safe: determinism, idempotent side effects, JSON-serializable inputs and outputs.
4. **Choose** between the Graph and Functional APIs for a given task, and write down your own rule of thumb.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Functional API overview](https://docs.langchain.com/oss/python/langgraph/functional-api) | `@entrypoint`, `@task`, injectable parameters (`previous`, `runtime`), `entrypoint.final`, serialization, determinism |
| [Use the Functional API](https://docs.langchain.com/oss/python/langgraph/use-functional-api) | Parallel tasks, calling graphs, retries, **resuming after an error**, short-term memory |
| [Choosing between Graph and Functional APIs](https://docs.langchain.com/oss/python/langgraph/choosing-apis) | The decision guide this lesson ends with |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · `@entrypoint` and `@task`

The Functional API adds LangGraph's features (checkpoints, streaming, interrupts) to **ordinary functions**. No state schema, no edges: `if`, `for` and function calls are the control flow.

- `@task` marks a unit of work. Calling it returns a **future**; `.result()` waits for the value. Tasks started before you call `.result()` run **in parallel**.
- `@entrypoint` marks the workflow. It takes **one** input (use a dict for several values) and is a runnable: `invoke`, `stream`, `batch`.

In [ ]:
# 🧪 Offline: Project 07's pipeline shape in ~20 lines of plain Python.
import time

from langgraph.func import entrypoint, task


@task
def plan(topic: str) -> list[str]:
    return [f"{topic}: basics", f"{topic}: tips", f"{topic}: troubleshooting"]


@task
def research(question: str) -> str:
    time.sleep(0.3)                                   # a slow search
    return f"notes on {question}"


@task
def write(topic: str, notes: list[str]) -> str:
    return f"# {topic}\n" + "\n".join(f"- {n}" for n in notes)


@entrypoint()
def pipeline(topic: str) -> str:
    questions = plan(topic).result()
    futures = [research(q) for q in questions]        # all three start now...
    notes = [f.result() for f in futures]             # ...and we wait for all of them here
    return write(topic, notes).result()


start = time.perf_counter()
print(pipeline.invoke("Battery care in winter"))
print(f"{time.perf_counter() - start:.1f}s for 3 × 0.3s of research")

**You should now see** a title and three bullet lines, then `0.3s for 3 × 0.3s of research` (parallel, like `Send` in 07.03).

### What just happened?

`pipeline` is a compiled LangGraph graph (`type(pipeline)` is `Pregel`, the same runtime as `StateGraph(...).compile()`). Each `@task` call became a step LangGraph can **record**. No checkpointer yet, so nothing was saved: Part 2 changes that.

In [ ]:
# 🧪 Offline: streaming works too. "updates" shows one line per finished task, then the entrypoint's result.
for update in pipeline.stream("Mounting outdoors", stream_mode="updates"):
    print(update)

**You should now see** `{'plan': [...]}`, three `{'research': 'notes on ...'}` lines, `{'write': '# Mounting outdoors...'}`, and `{'pipeline': '...'}` last.

---
## Part 2 · Checkpoints: resume after an error without redoing work

With a checkpointer, every finished task's result is **saved** under the run's `thread_id`. If a later task fails, invoke again with `None` on the same thread: finished tasks are **not re-run**, their saved results are replayed. That's durable execution (Module 08 goes deeper).

In [ ]:
# 🧪 Offline: a flaky step fails once. The expensive step before it must not run twice.
from langgraph.checkpoint.memory import InMemorySaver

CALLS = {"expensive": 0, "flaky": 0}
FAIL_ONCE = {"left": 1}


@task
def expensive_summary(text: str) -> str:
    CALLS["expensive"] += 1                          # imagine 5,000 tokens
    return text.upper()


@task
def flaky_publish(summary: str) -> str:
    CALLS["flaky"] += 1
    if FAIL_ONCE["left"]:
        FAIL_ONCE["left"] -= 1
        raise ConnectionError("publish API timed out")
    return f"published: {summary}"


@entrypoint(checkpointer=InMemorySaver())
def publish(text: str) -> str:
    return flaky_publish(expensive_summary(text).result()).result()


config = {"configurable": {"thread_id": "post-1"}}
try:
    publish.invoke("rain gauge tips", config)
except ConnectionError as exc:
    print("first run failed:", exc)
print(publish.invoke(None, config))                  # resume: same thread, input None
print(CALLS)

**You should now see** `first run failed: publish API timed out`, then `published: RAIN GAUGE TIPS`, then `{'expensive': 1, 'flaky': 2}`: the expensive task ran **once**; only the failed one ran again.

**Short-term memory: `previous` and `entrypoint.final`.** An entrypoint can receive what it returned last time on the same thread (`previous`). `entrypoint.final(value=..., save=...)` returns one thing and saves another.

In [ ]:
# 🧪 Offline: a running total per thread. `previous` = what was saved last time on this thread.
from typing import Any


@entrypoint(checkpointer=InMemorySaver())
def tally(amount: float, *, previous: Any = None) -> entrypoint.final[str, float]:
    total = (previous or 0.0) + amount
    return entrypoint.final(value=f"added {amount}, total {total}", save=total)


ana = {"configurable": {"thread_id": "ana"}}
ben = {"configurable": {"thread_id": "ben"}}
print(tally.invoke(89.0, ana))
print(tally.invoke(349.0, ana))
print(tally.invoke(10.0, ben))

**You should now see** `added 89.0, total 89.0`, `added 349.0, total 438.0`, and `added 10.0, total 10.0` for Ben: each thread has its own memory.

---
## Part 3 · The rules that make resuming safe

On resume, LangGraph **re-runs the entrypoint function from the top** and replays saved task results in order. That only works if the code takes the same path again:

| Rule | Why | Do |
|---|---|---|
| **Determinism** | Replay matches results to calls by order | Put randomness, time and I/O **inside tasks**, not in the entrypoint body |
| **Idempotent side effects** | A task may run again if it failed midway | Idempotency keys for payments/emails; check-then-write |
| **JSON-serializable** inputs/outputs | They are saved in the checkpoint | Dicts, lists, strings, numbers; Pydantic models via `.model_dump()` |

In [ ]:
# 🧪 Offline: why time/randomness must live in a task. Two "same" runs take different paths otherwise.
import random


@task
def pick_variant() -> str:                       # ✅ random inside a task: saved, replayed on resume
    return random.choice(["A", "B"])


@entrypoint(checkpointer=InMemorySaver())
def experiment(user: str) -> str:
    variant = pick_variant().result()
    return f"{user} sees variant {variant}"


cfg = {"configurable": {"thread_id": "exp-1"}}
first = experiment.invoke("ana", cfg)
state = experiment.get_state(cfg)
print(first, "| saved:", state.values)

**You should now see** `ana sees variant A` (or B) and the same text in `saved:`. If `random.choice` sat directly in `experiment`, a resumed run could pick the other variant and the replayed task results would no longer match the path: subtle bugs, no error.

---
## Part 4 · Graph or Functional?

The same pipeline in both APIs:

| | Graph API (07.02–07.04) | Functional API (this lesson) |
|---|---|---|
| Control flow | Edges, conditional edges, `Send`, `Command` | `if`, `for`, function calls |
| State | Explicit schema shared by nodes, reducers | Function-scoped variables; `previous` for memory |
| Visualization | Full graph drawing, Studio shows each node | One box per entrypoint (tasks don't appear as nodes) |
| Checkpoints | After every super-step | After every task |
| Good at | Branching, parallel paths that merge, team-readable designs | Adding persistence to existing code, linear flows, prototypes |
| Both | Streaming, interrupts (Module 08), the same runtime. You can call a graph from a task, or an entrypoint from a node |

In [ ]:
# 🧪 Offline: what you can draw. The Functional API shows one node; the Graph API shows the design.
print([n for n in pipeline.get_graph().nodes if not n.startswith("__")])

from mastery.graphs.content import build_content_graph, corpus_researcher, offline_models

m = offline_models()
project_graph = build_content_graph(m["planner"], m["writer"], m["critic"], corpus_researcher())
print([n for n in project_graph.get_graph().nodes if not n.startswith("__")])

**You should now see** `['pipeline']` and then Project 07's six nodes `['plan_outline', 'research_section', 'write_draft', 'critique', 'revise', 'finalize']`. That's a big reason Project 07 uses the Graph API (Studio shows every step), with the Functional version as its stretch goal.

**Your rule of thumb** (fill in, then copy to [`docs/research-log.md`](../../docs/research-log.md)):

> I use the **Graph API** when ______ (example from this module: ______).
> I use the **Functional API** when ______ (example: ______).
> I'd switch when ______.

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `TypeError` about multiple positional arguments | An entrypoint takes **one** input | Pass a dict: `invoke({"topic": t, "k": 3})` |
| Tasks run one after another | `.result()` called right after each call | Start all tasks first, then collect results |
| Resume re-runs everything | No checkpointer, or a different `thread_id` | `@entrypoint(checkpointer=...)` and the same config |
| `TypeError: Object of type X is not JSON serializable` | A non-serializable return value with a checkpointer | Return plain data (`.model_dump()`), keep clients/connections out of results |
| Resume takes a different path | Time/random/I/O in the entrypoint body | Move it into a `@task` |

## ⚠️ Production notes

> ⚠️ **Production note: `InMemorySaver` forgets everything on restart.**
> For real durability use a persistent checkpointer (SQLite locally, Postgres in production; Module 08), and keep task outputs small.

> ⚠️ **Production note: retries need idempotency.**
> `@task(retry_policy=RetryPolicy(max_attempts=3))` is one line (see the how-to). It's only safe if the task can run twice without harm.

## 🏋️ Exercises

1. **(Easy)** Make `pipeline` skip the "troubleshooting" question when the topic contains "basics".
2. **(Medium)** Add a `@task def review(draft) -> int` (score = number of bullet lines) and loop `write` → `review` at most 3 times in the entrypoint until the score ≥ 3.
3. **(Medium)** In Part 2, set `FAIL_ONCE["left"] = 1` again and resume on a **new** thread id. What does `CALLS` show, and why?
4. **(Stretch)** Call the compiled `project_graph` from inside a `@task` (`project_graph.invoke({"topic": t})`) and return only `["final"]`. When would mixing the APIs like this be useful?

<details>
<summary><b>Show solutions</b></summary>

**1.** In the entrypoint: `questions = [q for q in plan(topic).result() if not ("basics" in topic and "troubleshooting" in q)]`.

**2.** `for _ in range(3): draft = write(topic, notes).result(); if review(draft).result() >= 3: break`. Plain loops are the Functional API's strength; the counter is the `range`.

**3.** Both counters go up by one more for `expensive` and two for `flaky` (fail + run): a new thread has no saved results, so nothing is replayed.

**4.** It's the docs' "combining both APIs": a graph for the complex, visual part; a functional wrapper for simple orchestration around it (batch jobs, a script that runs the graph on many topics).
</details>

## ✅ Checkpoint

- [ ] I can write an `@entrypoint` with parallel `@task`s.
- [ ] I resumed a failed run without re-running finished tasks.
- [ ] I can state the determinism, idempotency and serialization rules.
- [ ] I wrote my Graph-vs-Functional rule of thumb.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Choosing between Graph and Functional APIs](https://docs.langchain.com/oss/python/langgraph/choosing-apis) and log **your own** rule of thumb (the template above) in [`docs/research-log.md`](../../docs/research-log.md). This is the module's research checkpoint.

## Summary

- `@task` = recorded unit of work returning a future; `@entrypoint` = the workflow, one input, a runnable.
- With a checkpointer, finished tasks are replayed on resume; `previous`/`entrypoint.final` give per-thread memory.
- Keep the entrypoint deterministic, side effects idempotent, values JSON-serializable.
- Graph API for branching/visual designs; Functional API for plain-code workflows. They mix.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 07.06 · Application structure, local server, and Studio** → [`07-06-app-structure-and-studio.ipynb`](07-06-app-structure-and-studio.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)